In [ ]:
#import packages 
from pprint import pprint
import tensorflow as tf
import keras
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts
import aneurysm_3Dsegmentation_in_CTA.model_utility.configuration as conf
import aneurysm_3Dsegmentation_in_CTA.model_utility.metrics as metrics
import segmentation_models_3D as sm3

I0000 00:00:1780754597.431203   39107 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# loading data tensors
dataSource = "data/CTA nii" 

smallAneurysm , mediumAneurysm , largeAneurysm = tts.dataSplitPerSize(dataSource)

trainSet , testSet = tts.dataSplitPerSample(
    [
        smallAneurysm ,
        mediumAneurysm ,
        largeAneurysm
    ] ,
    testRatio=0.2 ,
    seed=42
)

imgTrainSet , labelTrainSet = tts.dataTensorLoading(trainSet)
imgTestSet  , labelTestSet = tts.dataTensorLoading(testSet)


In [ ]:
# check for image pairs
for name in zip(imgTrainSet , labelTrainSet) :
    print(name[0])
    print(name[1])
    print("============")
print("//////////////////////////////////////////////// test ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgTestSet , labelTestSet) :
    print(name[0])
    print(name[1])
    print("============")

In [22]:
# pipeline configuring

geo      = utl.randomGeo(p=1)
crop     = utl.volume_crop((128 , 128 , 128))
windower = window.randomMultiWindowStackig(
    default = (200 , 620) ,
    wlRange=(170 , 225) ,
    wwRange=(600 , 650) ,
    p_wl=1 ,
    p_ww=1
)

# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64])
def rimg(imgPath , labelPath) :
    return utl.read_img(imgPath , labelPath)
def read_img(img , label) :
    imglbl = rimg(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    img , label = geo.flip(
        img , 
        label
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label



In [ ]:
# dataloaders
dataloaderTrain = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet))
    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )

    .cache("myCacheTrain")
    .shuffle(buffer_size=100)
    .batch(batch_size=2)
    
    .map(
        rot ,
        num_parallel_calls=4
    )
    .map(
        windower.WindowStacking ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
)

dataloaderValid = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet))

    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )

    .cache("myCacheValid")
    .batch(batch_size=2)
    
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
)

In [ ]:
cnt=0
# vectorize data model testing

for data in dataloaderTrain.take(10) :
    print(data)
    print(data.shape)
    cnt+=1
    print(cnt)

In [ ]:
#loss
binaryFocalLoss = sm3.losses.binary_focal_loss
diceLoss        = sm3.losses.dice_loss 
weightedBinaryFocalDiceLoss = conf.WeightedSumOfLosses(binaryFocalLoss , diceLoss , alpha=0.8)

model = sm3.models.unet.Unet(
    backbone_name="seresnet18" , 
    input_shape=(128 , 128 , 128 , 1) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features=conf.encoderF_d4
)
# model unfreezing
model = conf.unfreeze_model(
    model , 
    conf.unfreeze34_border , 
    keras.src.layers.normalization.batch_normalization.BatchNormalization
)

In [ ]:
# compilation
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3) ,
    loss = weightedBinaryFocalDiceLoss ,
    metrics=[
        metrics.V_Recall ,
        metrics.dice ,
        metrics.HD
    ]
)

In [ ]:
# model training
history = model.fit(
    x=dataloaderTrain ,
    epochs=200. ,
    validation_data=dataloaderValid
)